In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import re

import pandas as pd
import requests
import wandb

In [ ]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [ ]:
os.makedirs("outputs/alignment", exist_ok=True)

# Table prep functions

In [ ]:
def format_mean_sem(mean, sem, decimals=3, scale=1.0, math_mode=True):
    """Format a single mean/SEM pair as a LaTeX string, with optional scaling."""
    if pd.isna(mean):
        return "--"
    m = mean * scale
    if pd.isna(sem):
        s = f"{m:.{decimals}f}"
    else:
        s = f"{m:.{decimals}f} $\\pm$ {sem * scale:.{decimals}f}"
    return f"{s}" if math_mode else s


def add_mean_sem_columns(df, metrics, math_mode=True):
    """
    metrics: dict mapping output_col_name -> dict with:
        mean_col, sem_col, decimals (int), scale (float, default 1.0)
    """
    df = df.copy()
    for out_col, spec in metrics.items():
        df[out_col] = [
            format_mean_sem(
                m,
                s,
                decimals=spec.get("decimals", 3),
                scale=spec.get("scale", 1.0),
                math_mode=math_mode,
            )
            for m, s in zip(df[spec["mean_col"]], df[spec["sem_col"]])
        ]
    return df


def get_latex(df, columns=None, escape=False):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)

    latex_table = df.to_latex(
        index=False,
        columns=columns,
        escape=escape,  # False: let \pm and $ pass through untouched
    )
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }
    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)
    return latex_table


def parse_modality(x, modalities):
    for k, v in modalities.items():
        if k in x:
            return v

# WANDB API parsing

In [ ]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"

In [ ]:
project = "s2bms_alignment"
results_df_path = "outputs/alignment/results_s2bms.csv"

# S2BMS

## csv log

In [ ]:
if os.path.exists(results_df_path):
    df = pd.read_csv(results_df_path)
    print(f"There are {len(df)} records already in results.")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
else:
    df = None

# Separate concept columns

In [ ]:
import json

# Collect columns into 3 categories per version
versions = {"v8": {}, "v11": {}, "v10": {}}

for k, v in versions.items():
    with open(f"data/s2bms/concept_captions/{k}.json", "r") as f:
        data = json.load(f)

    v["in_dist_valid"] = []
    v["in_dist_unvalid"] = []
    v["out_dist"] = []

    for d in data:
        if "test" in d["col"]:
            name = d["col"].replace("test-", "test_dyn_k_index_")
            name = f"{name.replace('aux_', '')}_{'max' if d['is_max'] else 'min'}"
            v["out_dist"].append(name)
        else:
            name = f"{d['col'].replace('aux_', 'test_dyn_k_index_')}_{'max' if d['is_max'] else 'min'}"
            if d["val_av"]:
                v["in_dist_valid"].append(name)
            else:
                v["in_dist_unvalid"].append(name)

In [ ]:
versions["v11"]["in_dist_valid"]

## fetching

In [ ]:
def log_concept_avr(run, cols, name):
    metrics = dict(run.summary)
    matched_metrics = [metrics[c] for c in cols]
    value = sum(matched_metrics) / len(matched_metrics)
    run.summary.update({name: value})
    print(name, value)


def flatten(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    """Flatten a nested config dict (e.g. a Hydra config) into dot-separated keys."""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

In [ ]:
# project = "s2bms_vlm_baselines"

In [ ]:
api = wandb.Api()
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if run.state != "finished":
        print(f"{run.id} is not finished.")
        continue
    concept_v = run.config["data"]["caption_builder"]["concepts_fname"].replace(".json", "")
    if concept_v == "${version}":
        concept_v = run.summary["experiment"]
        match = re.search(r"(?<=_)v\d+", concept_v)
        concept_v = match.group() if match else None

    if run.summary.get("test_avr_top-dyn_k_index_in_dist_valid") is None:
        try:
            in_dist_valid = versions[concept_v]["in_dist_valid"]
            in_dist_unvalid = versions[concept_v]["in_dist_unvalid"]
            out_dist = versions[concept_v]["out_dist"]

            log_concept_avr(run, in_dist_valid, "test_avr_top-dyn_k_index_in_dist_valid")
            log_concept_avr(run, in_dist_unvalid, "test_avr_top-dyn_k_index_in_dist_unvalid")
            log_concept_avr(run, out_dist, "test_avr_top-dyn_k_index_ood")
        except Exception as e:
            if concept_v == "v8":
                print(f"v8 issue {run.id}: {e}")
            else:
                print(f"{run.id}: {e}")

    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue

    temp = dict(run.summary)
    config_dict = flatten(run.config)
    captures = {}
    captures["experiment"] = run.summary["experiment"]
    captures["run_id"] = run.id
    for k, v in temp.items():
        # if "test" not in k and "best" not in k:
        #     captures[f"best_{k}"] = v
        # else:
        #     captures[k] = v
        if "test" in k and "dyn_k_index" in k:
            captures[k] = v

    captures = {**captures, **config_dict}
    del captures["tags"]

    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
runs_df = pd.DataFrame(run_list)
df = None
if df is not None:
    runs_df = pd.concat([df, runs_df], ignore_index=True)
# runs_df.to_csv("outputs/alignment/results_s2bms.csv", index=False)

In [ ]:
cols_drop = []
for c in runs_df.columns:
    if any([isinstance(x, list) for x in runs_df[c]]):
        cols_drop.append(c)
    elif runs_df[c].nunique() <= 1:
        cols_drop.append(c)

runs_df.drop(columns=cols_drop, inplace=True)
runs_df

In [ ]:
## Selection
runs_sel = runs_df.copy()
runs_sel = runs_sel[runs_sel["data.caption_builder.concepts_fname"] == "v11.json"]
runs_sel = runs_sel[runs_sel["task_name"] == "train"]

for c in [
    "data.dataset.use_unlabelled_data",
    "data.batch_size",
    "seed",
    "model.loss_fn.sigma",
    "optimizer.lr",
]:
    print(c, runs_sel[c].unique())

use_unlabelled_data = True
runs_sel = runs_sel[runs_sel["optimizer.lr"] == 0.001]
runs_sel = runs_sel[runs_sel["data.dataset.use_unlabelled_data"] == use_unlabelled_data]

cols_drop = []
for c in runs_sel.columns:
    if any([isinstance(x, list) for x in runs_sel[c]]):
        cols_drop.append(c)
    elif runs_sel[c].nunique() <= 1:
        cols_drop.append(c)

runs_sel.drop(columns=cols_drop, inplace=True)

## average across seeds, pivot batch size vs sigma
runs_sel = (
    runs_sel.groupby(["data.batch_size", "model.loss_fn.sigma"])
    .agg(
        {
            "test_avr_top-dyn_k_index_in_dist_valid": ["mean", "sem"],
            "test_avr_top-dyn_k_index_in_dist_unvalid": ["mean", "sem"],
            "test_avr_top-dyn_k_index_ood": ["mean", "sem"],
        }
    )
    .reset_index()
)

## Plot the average of each index as a 2D heatmap of batch size vs sigma

import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 4, figsize=(20, 3))

for i_plot, name_index in enumerate(
    [
        "test_avr_top-dyn_k_index_in_dist_valid",
        "test_avr_top-dyn_k_index_in_dist_unvalid",
        "test_avr_top-dyn_k_index_ood",
        "average",
    ]
):
    if name_index == "average":
        runs_sel["average"] = runs_sel[
            [
                "test_avr_top-dyn_k_index_in_dist_valid",
                "test_avr_top-dyn_k_index_in_dist_unvalid",
                "test_avr_top-dyn_k_index_ood",
            ]
        ].mean(axis=1)
        df_plot = runs_sel.pivot(
            index="data.batch_size", columns="model.loss_fn.sigma", values=name_index
        )
    else:
        df_plot = runs_sel.pivot(
            index="data.batch_size", columns="model.loss_fn.sigma", values=(name_index, "mean")
        )
    sns.heatmap(df_plot, annot=True, fmt=".2f", ax=ax[i_plot])
    ax[i_plot].set_title(name_index, fontsize=10)
    ax[i_plot].set_xlabel("Sigma")
    ax[i_plot].set_ylabel("Batch Size")

fig.suptitle(
    f"Average of each index across seeds (use_unlabelled_data={use_unlabelled_data})",
    fontsize=10,
    y=1.05,
)

In [ ]:
runs_df["data.saved_split_file_name"].unique()

## Create DF with relevant info

In [ ]:
cols_keep = ["experiment", "seed", "run_id"]
for c in runs_df.columns:
    if c in cols_keep:
        continue
    if "test" in c and "dyn_k_index" in c:
        cols_keep.append(c)


df_all = runs_df[cols_keep]

if project == "s2bms_alignment":
    name_split = df_all["experiment"].str.split("_", expand=True)
    # df_all['']

name_split[11].unique()

# Plotting

In [ ]:
for col in runs_df.columns:
    print(col)

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "test_avr_top-dyn_k_index_in_dist_valid": "In distribution (present)",
    "test_avr_top-dyn_k_index_in_dist_unvalid": "In distribution (absent)",
    "test_avr_top-dyn_k_index_ood": "Out-of-distribution",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_avr_top-dyn_k_index_in_dist_valid", ascending=False)
summary.reset_index(inplace=True)

In [ ]:
summary